## DATASET STUDY

Study of the multilingual dataset used for the cross-lingual transferability study based on geometrical similarities of the embedding space.

#### SETUP

In [3]:
# setup
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

/gpfs/projects/bsc14/mouhida/TFG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset being used is MULTICLINAI

In [4]:
# paths of the dataset
data_path = Path("/gpfs/projects/bsc14/mouhida/datasets/multiclinai")
train_path = data_path / "train/MultiClinNER"
test_path = data_path / "test"

json_path = Path("/gpfs/projects/bsc14/mouhida/TFG/data/texts.json")

To do this study, we can use the tokenizer from a multilingual model like xlm-roberta-base

In [5]:
model_path = Path("/gpfs/projects/bsc14/mouhida/models/tfg/xlm-roberta-base")
tokenizer = AutoTokenizer.from_pretrained(model_path)

#### DATASET FILES

Pull all relevant files and see number

In [6]:
# list all train files (those finished in .txt from MutlicliNER)
train_files = sorted(train_path.rglob("*.txt"))

# list all test files 
# first obtain file names of the test set (as opposed to the background set)
test_map_df = pd.read_csv(test_path / "MultiClinAI_filenames_mapping_test.tsv", sep = "\t")
test_task_fnames = set(test_map_df.loc[test_map_df['test_or_background'] == "test", "task_fname"].to_list())
# pull full file paths for those in the test set
test_files = [f for f in test_path.rglob("*.txt") if f.stem in test_task_fnames]

# print lens
print(f"TRAIN LENGTH: {len(train_files)}")
print(f"TEST LENGTH: {len(test_files)}")

TRAIN LENGTH: 26418
TEST LENGTH: 13077


In [17]:
en, it, es = "-en-", "-it-", "-es-"

en_files = []
it_files = []
es_files = []

for f in test_files:
    if en in str(f):
        en_files.append(f)
    if it in str(f):
        it_files.append(f)
    if es in str(f):
        es_files.append(f)

In [23]:
ls = [en, it, es]

for i in range(0, 3):
    print(ls[i])

-en-
-it-
-es-


Build a df with all files to ease futher analysis:

In [8]:
# # empty list to save all files 
# records = []

# # iter through train files and fill in df
# for f in train_files:
#     fname = f.stem
#     lang = str(f.parents[3])[-2:]
#     dset = "train"
#     text = f.read_text(encoding = "utf-8")
#     records.append({"fname": fname, "lang": lang, "dset": dset, "text": text})
    
# # fill in for test files
# for f in test_files:
#     fname = f.stem
#     lang = str(f.parents[3])[-2:]
#     dset = "test"
#     text = f.read_text(encoding = "utf-8")
#     records.append({"fname": fname, "lang": lang, "dset": dset, "text": text})

# # create df with all data
# df = pd.DataFrame.from_records(records)

# # save json for further analysis
# json_path = Path("/gpfs/projects/bsc14/mouhida/TFG/data/texts.json")
# df.to_json(json_path, orient = 'records', lines = True)

#### TEXT CHARACTERISTICS

Look at different characteristics of teh dataset

In [9]:
df = pd.read_json(json_path, orient = 'records', lines = True)

CHARACTER LENGTH

In [10]:
df['text_len'] = df['text'].str.len()

print("Text character length stats:")
print(df['text_len'].describe().round(2))

Text character length stats:
count    39495.00
mean      3359.89
std       2540.91
min        104.00
25%       1544.00
50%       2574.00
75%       4567.50
max      17670.00
Name: text_len, dtype: float64


TOKEN LENGTH

In [11]:
# # tokenize all texts (batched) and compute the subword length per document
# token_lens = []
# batch = 256
# texts = df['text'].tolist()
# for i in range(0, len(texts), batch):
#     enc = tokenizer(texts[i:i+batch], add_special_tokens=False)
#     token_lens.extend(len(ids) for ids in enc['input_ids'])
# df['token_len'] = token_lens

# print("Text token length stats:")
# print(df['token_len'].describe().round(2))